# 1. Parse the Vivino collection

Normalize the two collection rounds into vintage, parent-wine, grape, and award tables. Requires the original merged API responses in `data/raw/vivino/`. This notebook does not perform the later cross-source merge; see `docs/reproducibility.md`.

**Recorded outputs:** figures and metrics are preserved from the course project. They have not been regenerated with the reorganized code. Datasets and fitted models are not included.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout whether Jupyter starts in the root or notebooks/.
PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "src/wine_quality").is_dir()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
from wine_quality.paths import RAW_DATA, INTERIM_DATA, PROCESSED_DATA, FIGURES

for folder in (FIGURES / "eda", FIGURES / "imputation", FIGURES / "models" / "shap"):
    folder.mkdir(parents=True, exist_ok=True)

(INTERIM_DATA / "vivino").mkdir(parents=True, exist_ok=True)


## Phase 2: Parsing the scrapped data

#### files with datas
- all_merged_round1.json  : wine info from wines explore pages
- all_merged_round2.json  : wine info from other vintages years, used winery
- winery_merged.json      : winery ids and info, needed to get other vintages years
- /countries              : files that contains scrapped info from explore pages, with filters

We will use all_merged* files to get all of the wines info, clean the data (remove useless data) and save under csv

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

### Extracting first round of data (~35 000 wines)

In [2]:
file = RAW_DATA / "vivino/all_merged_round1.json"

In [3]:
with open(file) as f:
    raw_js_wines = json.load(f)

In [10]:
raw_js_wines.keys(), len(raw_js_wines['vintage'])

(dict_keys(['vintage']), 35146)

In [ ]:
# structure of a wine
raw_js_wines['vintage'][0].keys()

dict_keys(['id', 'seo_name', 'name', 'statistics', 'organic_certification_id', 'certified_biodynamic', 'image', 'wine', 'description', 'wine_critic_reviews', 'awards', 'ratings_distribution', 'ranking', 'recommended_drinking_window', 'wine_facts', 'winemaker', 'wine_maker', 'grape_composition', 'year', 'grapes', 'has_valid_ratings'])


In [36]:
raw_js_wines['vintage'][0]["has_valid_ratings"]

True

In [39]:
# Revelent keys to keep

# seo_name/name, which to keep ?
# wine is about the general wine without taking account the years
# might need award?  see if keep
# has_valid_rating see if keep

keep_keys = ['id', "seo_name", "name", "statistics", "certified_biodynamic", "wine", "description", "awards", "ratings_distribution","ranking", "recommended_drinking_window", 
            "wine_facts", "grape_composition", "year", "grapes", "has_valid_ratings"]

# Keys not to keep

# organic certification id: need to have access to that
# image : bunch of urls
# wine_critic_reviews : gives only critics
# winemaker: no need for it
# wine_maker: no need for it

In [38]:
raw_wines = raw_js_wines['vintage']

In [43]:
raw_filtered_wines = [{k: wine[k] for k in keep_keys} for wine in raw_wines]

In [45]:
raw_filtered_wines[0].keys()

dict_keys(['id', 'seo_name', 'name', 'statistics', 'certified_biodynamic', 'wine', 'description', 'awards', 'ratings_distribution', 'ranking', 'recommended_drinking_window', 'wine_facts', 'grape_composition', 'year', 'grapes', 'has_valid_ratings'])

#### temporarily saving

In [46]:
with open(INTERIM_DATA / "vivino/filtered_wines_round1.json", "w") as f:
    json.dump(raw_filtered_wines, f, indent=2)

In [270]:
with open(INTERIM_DATA / "vivino/filtered_wines_round1.json") as f:
    round1 = json.load(f)

In [273]:
df_round1 = pd.DataFrame(round1) 

#### saving for round 2

In [51]:
file2 = RAW_DATA / "vivino/all_merged_round2.json"

In [52]:
with open(file2) as f:
    raw_js_wines2 = json.load(f)

In [53]:
raw_wines2 = raw_js_wines2['vintage']
raw_filtered_wines2 = [{k: wine[k] for k in keep_keys} for wine in raw_wines2]

In [54]:
with open(INTERIM_DATA / "vivino/filtered_wines_round2.json", "w") as f:
    json.dump(raw_filtered_wines2, f, indent=2)

## CSV parsing 

In [ ]:
# in the structure, there are references to ids, separate into many tables
# wine id
# grape id
# winery

In [71]:
df_round1 = pd.DataFrame(raw_filtered_wines)

Cols to split:
- wine
- ratings_distribution -> liste [1, 2, 3, 4, 5] stars

In [274]:
df_round1

,id,seo_name,name,statistics,certified_biodynamic,wine,description,awards,ratings_distribution,ranking,recommended_drinking_window,wine_facts,grape_composition,year,grapes,has_valid_ratings
0,100015094,chateau-mouton-rothschild-aile-d-argent-blanc-...,Château Mouton Rothschild Aile d'Argent Blanc ...,"{'status': 'Normal', 'ratings_count': 381, 'ra...",False,"{'id': 2284844, 'name': 'Aile d'Argent Blanc B...",,[],"{'1': 3, '2': 3, '3': 67, '4': 246, '5': 62}",{'country': {'description': 'Rank within Franc...,"{'start_year': 2018, 'end_year': 2026, 'status...","{'alcohol': 13.5, 'oak_aged': 0, 'certified_bi...","{'17': 0, '18': 0}",2016,"[{'id': 17, 'name': 'Sauvignon Blanc', 'seo_na...",True
1,10013618,reynvaan-the-classic-estate-cabernet-sauvignon...,Reynvaan The Classic Estate Cabernet Sauvignon...,"{'status': 'BelowThreshold', 'ratings_count': ...",None,"{'id': 2369586, 'name': 'The Classic Estate Ca...",,[],"{'1': 0, '2': 2, '3': 8, '4': 8, '5': 3}",{'country': {'description': 'Rank within Unite...,"{'start_year': 2015, 'end_year': 2028, 'status...",{},{},2013,None,False
2,10023048,j-m-gremillet-blanc-de-blancs-brut-champagne-nv,Gremillet Blanc de Blancs Brut Champagne,"{'status': 'Normal', 'ratings_count': 486, 'ra...",None,"{'id': 2714706, 'name': 'Blanc de Blancs Brut ...",,"[{'ID': 35171, 'CompetitionID': 1, 'VintageID'...","{'1': 5, '2': 10, '3': 161, '4': 270, '5': 40}",{'country': {'description': 'Rank within Franc...,"{'start_year': None, 'end_year': None, 'status...","{'alcohol': 12.5, 'residual_sugar_grams_per_li...",{'5': 100},N.V.,"[{'id': 5, 'name': 'Chardonnay', 'seo_name': '...",True
3,10027177,deutz-blanc-de-blancs-brut-champagne-2013,Deutz Blanc de Blancs Brut Champagne 2013,"{'status': 'Normal', 'ratings_count': 373, 'ra...",None,"{'id': 1142211, 'name': 'Blanc de Blancs Brut ...",,[],"{'1': 0, '2': 2, '3': 57, '4': 274, '5': 40}",{'country': {'description': 'Rank within Franc...,"{'start_year': 2015, 'end_year': 2028, 'status...",{'alcohol': 12.5},{'5': 100},2013,"[{'id': 5, 'name': 'Chardonnay', 'seo_name': '...",True
4,100451553,livon-selezione-anniversario-2014,Livon Selezione Anniversario 2014,"{'status': 'Normal', 'ratings_count': 25, 'rat...",None,"{'id': 4153013, 'name': 'Selezione Anniversari...",,[],"{'1': 0, '2': 0, '3': 3, '4': 19, '5': 3}",{'country': {'description': 'Rank within Italy...,"{'start_year': 2016, 'end_year': 2024, 'status...",{'alcohol': 13.5},{},2014,None,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35141,99616661,chateau-du-haut-pick-sauternes-2014,Château du Haut Pick Sauternes 2014,"{'status': 'Normal', 'ratings_count': 42, 'rat...",None,"{'id': 1834572, 'name': 'Sauternes', 'seo_name...",,[],"{'1': 1, '2': 2, '3': 24, '4': 11, '5': 4}",{'country': {'description': 'Rank within Franc...,"{'start_year': 2018, 'end_year': 2034, 'status...",{},{},2014,None,True
35142,9986182,domaine-perrot-minot-vignes-centenaires-nuits-...,Domaine Perrot-Minot Vignes Centenaires Nuits-...,"{'status': 'Normal', 'ratings_count': 43, 'rat...",None,"{'id': 2711130, 'name': 'Vignes Centenaires Nu...",,[],"{'1': 0, '2': 1, '3': 0, '4': 31, '5': 11}",{'country': {'description': 'Rank within Franc...,"{'start_year': 2014, 'end_year': 2028, 'status...",{},{},2012,None,True
35143,99861,fr-chateau-pape-clement-pessac-leognan-grand-c...,Château Pape Clément Pessac-Léognan (Grand Cru...,"{'status': 'Normal', 'ratings_count': 1641, 'r...",None,"{'id': 82613, 'name': 'Pessac-Léognan (Grand C...",The grapes are initially sorted in the vineyar...,[],"{'1': 9, '2': 13, '3': 127, '4': 1012, '5': 480}",{'country': {'description': 'Rank within Franc...,"{'start_year': 2011, 'end_year': 2026, 'status...","{'alcohol': 13.5, 'residual_sugar_grams_per_li...",{},2006,None,True
35144,99949,biondi-santi-brunello-di-montalcino-riserva-2001,Biondi-Santi Brunello di Montalcino Riserva 2001,"{'status': 'Normal', 'ratings_count': 154, 'ra...",None,"{'id': 82698, 'name': 'Brunello di Montalcino ...",,

In [275]:
# table of grapes
grapes1 = df_round1['grapes'].values

In [ ]:
grapes1 = [[element[i] for i in range(len(element))] for element in grapes1 if element]

In [278]:
grapes1 = [el for sub in grapes1 for el in sub  if sub]

In [280]:
df_grapes1 = pd.DataFrame(grapes1)

In [281]:
df_grapes1 = df_grapes1.drop_duplicates()
df_grapes1

,id,name,seo_name,has_detailed_info,wines_count,parent_grape_id
0,17,Sauvignon Blanc,sauvignon-blanc,True,292440,NaN
1,18,Sémillon,semillion,True,65710,NaN
2,5,Chardonnay,chardonnay,True,604208,NaN
4,19,Tempranillo,tempranillo,True,172842,NaN
6,199,Pinot Nero,pinot-nero,False,16533,14.0
...,...,...,...,...,...,...
11545,245,Picpoul Blanc,picpoul-blanc,False,3522,NaN
11557,982,Fortana,fortana,False,183,NaN
11577,1551,Trebbiano D'Abruzzo,trebbiano-d-abruzzo,False,1905,NaN
11721,566,Morenillo,morenillo,False,159,NaN


In [282]:
# save table
df_grapes1.to_csv(INTERIM_DATA / "vivino/grapes1_df.csv", index=False)

In [284]:
# split col grapes
l = df_round1['grapes'].values
l = [[el[i]["id"] for i in range(len(el))] if el else np.nan for el in l]
df_round1['grapes'] = l

In [285]:
# table of wine (parent)
wine1 = df_round1["wine"].values
wine1

array([{'id': 2284844, 'name': "Aile d'Argent Blanc Bordeaux", 'seo_name': 'aile-d-argent-blanc-bordeaux', 'type_id': 2, 'vintage_type': 0, 'is_natural': False, 'region': {'id': 383, 'name': 'Bordeaux', 'name_en': '', 'seo_name': 'bordeaux', 'country': {'code': 'fr', 'name': 'France', 'native_name': 'France', 'seo_name': 'france', 'currency': {'code': 'EUR', 'name': 'Euros', 'prefix': '€', 'suffix': None}, 'regions_count': 1309, 'users_count': 7235204, 'wines_count': 452623, 'wineries_count': 69083, 'most_used_grapes': [{'id': 14, 'name': 'Pinot Noir', 'seo_name': 'pinot-noir', 'has_detailed_info': True, 'wines_count': 572334, 'parent_grape_id': None}, {'id': 5, 'name': 'Chardonnay', 'seo_name': 'chardonnay', 'has_detailed_info': True, 'wines_count': 604208, 'parent_grape_id': None}, {'id': 10, 'name': 'Merlot', 'seo_name': 'merlot', 'has_detailed_info': True, 'wines_count': 566719, 'parent_grape_id': None}]}, 'parent_id': None, 'background_image': {'location': '//images.vivino.com/reg

In [286]:
df_wines1 = pd.json_normalize(wine1, sep='_')
df_wines1 = df_wines1.drop_duplicates(subset="id")
df_wines1

,id,name,seo_name,type_id,vintage_type,is_natural,vintage_mask,review_status,rank,hidden,...,style_region,style_region_background_image,specialists_notes,winery_dashboard_url,winery_background_image_location,winery_background_image_variations_large,winery_background_image_variations_medium,winery_background_image_variations_small,style_image_location,style
0,2284844,Aile d'Argent Blanc Bordeaux,aile-d-argent-blanc-bordeaux,2,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2369586,The Classic Estate Cabernet Sauvignon,the-classic-estate-cabernet-sauvignon,1,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2714706,Blanc de Blancs Brut Champagne,blanc-de-blancs-brut-champagne,3,1,False,[>0],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1142211,Blanc de Blancs Brut Champagne,blanc-de-blancs-brut-champagne,3,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4153013,Selezione Anniversario,selezione-anniversario,2,0,False,NaN,2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35110,2694726,Brut,brut,3,1,True,[>0],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35130,1203891,Cava AgustÍ-Vilaret Gran Reserva Extra Brut,cava-agusti-vilaret-gran-reserva-extra-brut,3,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35132,82079,The Complicator,the-complicator,1,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35133,82081,Left Field,left-field,1,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [287]:
# save table
df_wines1.to_csv(INTERIM_DATA / "vivino/wines1_df.csv", index=False)

In [288]:
# split col wine
l = df_round1['wine']
l = [el["id"] if el else np.nan for el in l]
df_round1['wine'] = l

In [289]:
df_round1

,id,seo_name,name,statistics,certified_biodynamic,wine,description,awards,ratings_distribution,ranking,recommended_drinking_window,wine_facts,grape_composition,year,grapes,has_valid_ratings
0,100015094,chateau-mouton-rothschild-aile-d-argent-blanc-...,Château Mouton Rothschild Aile d'Argent Blanc ...,"{'status': 'Normal', 'ratings_count': 381, 'ra...",False,2284844,,[],"{'1': 3, '2': 3, '3': 67, '4': 246, '5': 62}",{'country': {'description': 'Rank within Franc...,"{'start_year': 2018, 'end_year': 2026, 'status...","{'alcohol': 13.5, 'oak_aged': 0, 'certified_bi...","{'17': 0, '18': 0}",2016,"[17, 18]",True
1,10013618,reynvaan-the-classic-estate-cabernet-sauvignon...,Reynvaan The Classic Estate Cabernet Sauvignon...,"{'status': 'BelowThreshold', 'ratings_count': ...",None,2369586,,[],"{'1': 0, '2': 2, '3': 8, '4': 8, '5': 3}",{'country': {'description': 'Rank within Unite...,"{'start_year': 2015, 'end_year': 2028, 'status...",{},{},2013,NaN,False
2,10023048,j-m-gremillet-blanc-de-blancs-brut-champagne-nv,Gremillet Blanc de Blancs Brut Champagne,"{'status': 'Normal', 'ratings_count': 486, 'ra...",None,2714706,,"[{'ID': 35171, 'CompetitionID': 1, 'VintageID'...","{'1': 5, '2': 10, '3': 161, '4': 270, '5': 40}",{'country': {'description': 'Rank within Franc...,"{'start_year': None, 'end_year': None, 'status...","{'alcohol': 12.5, 'residual_sugar_grams_per_li...",{'5': 100},N.V.,[5],True
3,10027177,deutz-blanc-de-blancs-brut-champagne-2013,Deutz Blanc de Blancs Brut Champagne 2013,"{'status': 'Normal', 'ratings_count': 373, 'ra...",None,1142211,,[],"{'1': 0, '2': 2, '3': 57, '4': 274, '5': 40}",{'country': {'description': 'Rank within Franc...,"{'start_year': 2015, 'end_year': 2028, 'status...",{'alcohol': 12.5},{'5': 100},2013,[5],True
4,100451553,livon-selezione-anniversario-2014,Livon Selezione Anniversario 2014,"{'status': 'Normal', 'ratings_count': 25, 'rat...",None,4153013,,[],"{'1': 0, '2': 0, '3': 3, '4': 19, '5': 3}",{'country': {'description': 'Rank within Italy...,"{'start_year': 2016, 'end_year': 2024, 'status...",{'alcohol': 13.5},{},2014,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35141,99616661,chateau-du-haut-pick-sauternes-2014,Château du Haut Pick Sauternes 2014,"{'status': 'Normal', 'ratings_count': 42, 'rat...",None,1834572,,[],"{'1': 1, '2': 2, '3': 24, '4': 11, '5': 4}",{'country': {'description': 'Rank within Franc...,"{'start_year': 2018, 'end_year': 2034, 'status...",{},{},2014,NaN,True
35142,9986182,domaine-perrot-minot-vignes-centenaires-nuits-...,Domaine Perrot-Minot Vignes Centenaires Nuits-...,"{'status': 'Normal', 'ratings_count': 43, 'rat...",None,2711130,,[],"{'1': 0, '2': 1, '3': 0, '4': 31, '5': 11}",{'country': {'description': 'Rank within Franc...,"{'start_year': 2014, 'end_year': 2028, 'status...",{},{},2012,NaN,True
35143,99861,fr-chateau-pape-clement-pessac-leognan-grand-c...,Château Pape Clément Pessac-Léognan (Grand Cru...,"{'status': 'Normal', 'ratings_count': 1641, 'r...",None,82613,The grapes are initially sorted in the vineyar...,[],"{'1': 9, '2': 13, '3': 127, '4': 1012, '5': 480}",{'country': {'description': 'Rank within Franc...,"{'start_year': 2011, 'end_year': 2026, 'status...","{'alcohol': 13.5, 'residual_sugar_grams_per_li...",{},2006,NaN,True
35144,99949,biondi-santi-brunello-di-montalcino-riserva-2001,Biondi-Santi Brunello di Montalcino Riserva 2001,"{'status': 'Normal', 'ratings_count': 154, 'ra...",None,82698,,[],"{'1': 0, '2': 3, '3': 9, '4': 65, '5': 77}",{'country': {'description': 'Rank within Italy...,"{'start_year': 2011, 'end_year': 2028, 'status...","{'alcohol': 14, 'contains_sulfites_allergens':...",{'16': 100},2001,[16],True


In [290]:
# rating distributions
rat1 = df_round1['ratings_distribution']
rat1 = [[e[str(i)] for i in range(1, 6)] for e in rat1]
df_round1['ratings_distribution'] = rat1

In [291]:
df_round1

,id,seo_name,name,statistics,certified_biodynamic,wine,description,awards,ratings_distribution,ranking,recommended_drinking_window,wine_facts,grape_composition,year,grapes,has_valid_ratings
0,100015094,chateau-mouton-rothschild-aile-d-argent-blanc-...,Château Mouton Rothschild Aile d'Argent Blanc ...,"{'status': 'Normal', 'ratings_count': 381, 'ra...",False,2284844,,[],"[3, 3, 67, 246, 62]",{'country': {'description': 'Rank within Franc...,"{'start_year': 2018, 'end_year': 2026, 'status...","{'alcohol': 13.5, 'oak_aged': 0, 'certified_bi...","{'17': 0, '18': 0}",2016,"[17, 18]",True
1,10013618,reynvaan-the-classic-estate-cabernet-sauvignon...,Reynvaan The Classic Estate Cabernet Sauvignon...,"{'status': 'BelowThreshold', 'ratings_count': ...",None,2369586,,[],"[0, 2, 8, 8, 3]",{'country': {'description': 'Rank within Unite...,"{'start_year': 2015, 'end_year': 2028, 'status...",{},{},2013,NaN,False
2,10023048,j-m-gremillet-blanc-de-blancs-brut-champagne-nv,Gremillet Blanc de Blancs Brut Champagne,"{'status': 'Normal', 'ratings_count': 486, 'ra...",None,2714706,,"[{'ID': 35171, 'CompetitionID': 1, 'VintageID'...","[5, 10, 161, 270, 40]",{'country': {'description': 'Rank within Franc...,"{'start_year': None, 'end_year': None, 'status...","{'alcohol': 12.5, 'residual_sugar_grams_per_li...",{'5': 100},N.V.,[5],True
3,10027177,deutz-blanc-de-blancs-brut-champagne-2013,Deutz Blanc de Blancs Brut Champagne 2013,"{'status': 'Normal', 'ratings_count': 373, 'ra...",None,1142211,,[],"[0, 2, 57, 274, 40]",{'country': {'description': 'Rank within Franc...,"{'start_year': 2015, 'end_year': 2028, 'status...",{'alcohol': 12.5},{'5': 100},2013,[5],True
4,100451553,livon-selezione-anniversario-2014,Livon Selezione Anniversario 2014,"{'status': 'Normal', 'ratings_count': 25, 'rat...",None,4153013,,[],"[0, 0, 3, 19, 3]",{'country': {'description': 'Rank within Italy...,"{'start_year': 2016, 'end_year': 2024, 'status...",{'alcohol': 13.5},{},2014,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35141,99616661,chateau-du-haut-pick-sauternes-2014,Château du Haut Pick Sauternes 2014,"{'status': 'Normal', 'ratings_count': 42, 'rat...",None,1834572,,[],"[1, 2, 24, 11, 4]",{'country': {'description': 'Rank within Franc...,"{'start_year': 2018, 'end_year': 2034, 'status...",{},{},2014,NaN,True
35142,9986182,domaine-perrot-minot-vignes-centenaires-nuits-...,Domaine Perrot-Minot Vignes Centenaires Nuits-...,"{'status': 'Normal', 'ratings_count': 43, 'rat...",None,2711130,,[],"[0, 1, 0, 31, 11]",{'country': {'description': 'Rank within Franc...,"{'start_year': 2014, 'end_year': 2028, 'status...",{},{},2012,NaN,True
35143,99861,fr-chateau-pape-clement-pessac-leognan-grand-c...,Château Pape Clément Pessac-Léognan (Grand Cru...,"{'status': 'Normal', 'ratings_count': 1641, 'r...",None,82613,The grapes are initially sorted in the vineyar...,[],"[9, 13, 127, 1012, 480]",{'country': {'description': 'Rank within Franc...,"{'start_year': 2011, 'end_year': 2026, 'status...","{'alcohol': 13.5, 'residual_sugar_grams_per_li...",{},2006,NaN,True
35144,99949,biondi-santi-brunello-di-montalcino-riserva-2001,Biondi-Santi Brunello di Montalcino Riserva 2001,"{'status': 'Normal', 'ratings_count': 154, 'ra...",None,82698,,[],"[0, 3, 9, 65, 77]",{'country': {'description': 'Rank within Italy...,"{'start_year': 2011, 'end_year': 2028, 'status...","{'alcohol': 14, 'contains_sulfites_allergens':...",{'16': 100},2001,[16],True


In [292]:
l = df_round1["awards"].values
l = [subel for el in l for subel in el if subel]
l

[{'ID': 35171,
  'CompetitionID': 1,
  'VintageID': 10023048,
  'Edition': '2014',
  'Medal': 'Silver',
  'Trophy': None,
  'Score': None,
  'Visible': True,
  'URL': 'http://awards.decanter.com/DWWA/2014/Wine/53202?name=Gremillet-Blanc%20de%20Blancs%20Brut-NV',
  'Origin': 0,
  'UserID': None,
  'UpdatedAt': '2022-10-19T12:44:26Z',
  'CreatedAt': '2021-01-13T09:15:58Z'},
 {'ID': 57122,
  'CompetitionID': 1,
  'VintageID': 10023048,
  'Edition': '2015',
  'Medal': 'Commended',
  'Trophy': None,
  'Score': None,
  'Visible': True,
  'URL': 'http://awards.decanter.com/DWWA/2015/Wine/67124?name=Gremillet-Blanc%20de%20Blancs%20Brut-NV',
  'Origin': 0,
  'UserID': None,
  'UpdatedAt': '2022-10-19T12:49:52Z',
  'CreatedAt': '2021-01-13T09:20:34Z'},
 {'ID': 4175,
  'CompetitionID': 4,
  'VintageID': 10023048,
  'Edition': '2015',
  'Medal': 'Silver',
  'Trophy': None,
  'Score': None,
  'Visible': True,
  'URL': 'https://www.iwsc.net/result/detail/62445',
  'Origin': 0,
  'UserID': None,
  'U

In [293]:
df_awards1 = pd.json_normalize(l, sep='_')
df_awards1 = df_awards1.drop_duplicates(subset="ID")
df_awards1

,ID,CompetitionID,VintageID,Edition,Medal,Trophy,Score,Visible,URL,Origin,UserID,UpdatedAt,CreatedAt
0,35171,1,10023048,2014,Silver,None,None,True,http://awards.decanter.com/DWWA/2014/Wine/5320...,0,NaN,2022-10-19T12:44:26Z,2021-01-13T09:15:58Z
1,57122,1,10023048,2015,Commended,None,None,True,http://awards.decanter.com/DWWA/2015/Wine/6712...,0,NaN,2022-10-19T12:49:52Z,2021-01-13T09:20:34Z
2,4175,4,10023048,2015,Silver,None,None,True,https://www.iwsc.net/result/detail/62445,0,NaN,2022-10-19T12:36:26Z,2021-01-15T17:33:33Z
3,87134,7,10023048,2015,Bronze,None,None,True,http://awards.decanter.com/DAWA/2015/Wine/7676...,0,NaN,2022-10-19T12:57:21Z,2021-01-13T09:28:33Z
4,93077,7,10023048,2017,Bronze,None,88,True,http://awards.decanter.com/DAWA/2017/Wine/4064...,0,NaN,2022-10-19T12:58:48Z,2021-01-13T09:29:57Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1581,94689,6,91756680,Summer Tasting 2017,Gold,None,None,True,https://www.meininger.de/en/wine/2016-fransola...,0,NaN,2022-10-19T12:59:12Z,2018-03-15T12:40:16Z
1582,27453,1,9611919,2014,Silver,None,None,True,http://awards.decanter.com/DWWA/2014/Wine/5396...,0,NaN,2022-10-19T12:42:27Z,2021-01-13T09:14:18Z
1583,36846,2,9611919,2014,Bronze,None,None,True,https://www.internationalwinechallenge.com/can...,0,NaN,2022-10-19T12:44:51Z,2021-01-16T19:56:50Z
1584,142435,1,98578275,2017,Gold,None,95,True,http://awards.decanter.com/DWWA/2017/Wine/3891...,0,NaN,2022-10-19T13:11:12Z,2018-03-23T13:06:24Z


In [294]:
# save table
df_awards1.to_csv(INTERIM_DATA / "vivino/awards1_df.csv", index=False)

In [295]:
l = df_round1["awards"]
l = [[el[i]["ID"] for i in range(len(el))] if el else np.nan for el in l]
l

[nan,
 nan,
 [35171, 57122, 4175, 87134, 93077],
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 [55799],
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 [154205, 129599, 105440],
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 [45476],
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan

In [296]:
df_round1["awards"] = l
df_round1

,id,seo_name,name,statistics,certified_biodynamic,wine,description,awards,ratings_distribution,ranking,recommended_drinking_window,wine_facts,grape_composition,year,grapes,has_valid_ratings
0,100015094,chateau-mouton-rothschild-aile-d-argent-blanc-...,Château Mouton Rothschild Aile d'Argent Blanc ...,"{'status': 'Normal', 'ratings_count': 381, 'ra...",False,2284844,,NaN,"[3, 3, 67, 246, 62]",{'country': {'description': 'Rank within Franc...,"{'start_year': 2018, 'end_year': 2026, 'status...","{'alcohol': 13.5, 'oak_aged': 0, 'certified_bi...","{'17': 0, '18': 0}",2016,"[17, 18]",True
1,10013618,reynvaan-the-classic-estate-cabernet-sauvignon...,Reynvaan The Classic Estate Cabernet Sauvignon...,"{'status': 'BelowThreshold', 'ratings_count': ...",None,2369586,,NaN,"[0, 2, 8, 8, 3]",{'country': {'description': 'Rank within Unite...,"{'start_year': 2015, 'end_year': 2028, 'status...",{},{},2013,NaN,False
2,10023048,j-m-gremillet-blanc-de-blancs-brut-champagne-nv,Gremillet Blanc de Blancs Brut Champagne,"{'status': 'Normal', 'ratings_count': 486, 'ra...",None,2714706,,"[35171, 57122, 4175, 87134, 93077]","[5, 10, 161, 270, 40]",{'country': {'description': 'Rank within Franc...,"{'start_year': None, 'end_year': None, 'status...","{'alcohol': 12.5, 'residual_sugar_grams_per_li...",{'5': 100},N.V.,[5],True
3,10027177,deutz-blanc-de-blancs-brut-champagne-2013,Deutz Blanc de Blancs Brut Champagne 2013,"{'status': 'Normal', 'ratings_count': 373, 'ra...",None,1142211,,NaN,"[0, 2, 57, 274, 40]",{'country': {'description': 'Rank within Franc...,"{'start_year': 2015, 'end_year': 2028, 'status...",{'alcohol': 12.5},{'5': 100},2013,[5],True
4,100451553,livon-selezione-anniversario-2014,Livon Selezione Anniversario 2014,"{'status': 'Normal', 'ratings_count': 25, 'rat...",None,4153013,,NaN,"[0, 0, 3, 19, 3]",{'country': {'description': 'Rank within Italy...,"{'start_year': 2016, 'end_year': 2024, 'status...",{'alcohol': 13.5},{},2014,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35141,99616661,chateau-du-haut-pick-sauternes-2014,Château du Haut Pick Sauternes 2014,"{'status': 'Normal', 'ratings_count': 42, 'rat...",None,1834572,,NaN,"[1, 2, 24, 11, 4]",{'country': {'description': 'Rank within Franc...,"{'start_year': 2018, 'end_year': 2034, 'status...",{},{},2014,NaN,True
35142,9986182,domaine-perrot-minot-vignes-centenaires-nuits-...,Domaine Perrot-Minot Vignes Centenaires Nuits-...,"{'status': 'Normal', 'ratings_count': 43, 'rat...",None,2711130,,NaN,"[0, 1, 0, 31, 11]",{'country': {'description': 'Rank within Franc...,"{'start_year': 2014, 'end_year': 2028, 'status...",{},{},2012,NaN,True
35143,99861,fr-chateau-pape-clement-pessac-leognan-grand-c...,Château Pape Clément Pessac-Léognan (Grand Cru...,"{'status': 'Normal', 'ratings_count': 1641, 'r...",None,82613,The grapes are initially sorted in the vineyar...,NaN,"[9, 13, 127, 1012, 480]",{'country': {'description': 'Rank within Franc...,"{'start_year': 2011, 'end_year': 2026, 'status...","{'alcohol': 13.5, 'residual_sugar_grams_per_li...",{},2006,NaN,True
35144,99949,biondi-santi-brunello-di-montalcino-riserva-2001,Biondi-Santi Brunello di Montalcino Riserva 2001,"{'status': 'Normal', 'ratings_count': 154, 'ra...",None,82698,,NaN,"[0, 3, 9, 65, 77]",{'country': {'description': 'Rank within Italy...,"{'start_year': 2011, 'end_year': 2028, 'status...","{'alcohol': 14, 'contains_sulfites_allergens':...",{'16': 100},2001,[16],True


In [297]:
df_round1_wind = pd.json_normalize(df_round1['recommended_drinking_window'], sep='_').add_prefix("window_")
df_round1_wind

,window_start_year,window_end_year,window_status
0,2018.0,2026.0,5.0
1,2015.0,2028.0,5.0
2,NaN,NaN,1.0
3,2015.0,2028.0,5.0
4,2016.0,2024.0,5.0
...,...,...,...
35141,2018.0,2034.0,5.0
35142,2014.0,2028.0,5.0
35143,2011.0,2026.0,5.0
35144,2011.0,2028.0,5.0


In [298]:
df_round1 = pd.concat([df_round1.drop(columns='recommended_drinking_window'), df_round1_wind], axis = 1)

In [299]:
df_stats = pd.json_normalize(df_round1['statistics'].values, sep='_').add_prefix("stats_")
df_stats

,stats_status,stats_ratings_count,stats_ratings_average,stats_labels_count,stats_reviews_count
0,Normal,381,4.2,3397,148
1,BelowThreshold,21,0.0,168,5
2,Normal,486,3.9,4389,150
3,Normal,373,4.2,2100,123
4,Normal,25,4.1,91,9
...,...,...,...,...,...
35141,Normal,42,3.5,326,17
35142,Normal,43,4.5,362,17
35143,Normal,1641,4.4,14485,499
35144,Normal,154,4.6,1081,62


In [300]:
df_round1 = pd.concat([df_round1.drop(columns='statistics'), df_stats], axis = 1)

In [306]:
df_round1.to_csv(INTERIM_DATA / "vivino/df_round1.csv", index=False)

---

In [190]:
# round 2 files parsing

In [212]:
with open(RAW_DATA / "vivino/all_merged_round2.json") as f:
    raw_js_wines = json.load(f)

In [213]:
raw_js_wines = raw_js_wines['vintage']

In [214]:
raw_filtered_wines = [{k: wine[k] for k in keep_keys} for wine in raw_js_wines]

In [215]:
with open(INTERIM_DATA / "vivino/filtered_wines_round2.json", "w") as f:
    json.dump(raw_filtered_wines, f, indent=2)

In [216]:
with open(INTERIM_DATA / "vivino/filtered_wines_round2.json") as f:
    raw_js_wines = json.load(f)

In [219]:
# parsing the data
df_round2 = pd.DataFrame(raw_js_wines)

In [221]:
grapes2 = df_round2['grapes'].values
grapes2 = [element[0] for element in grapes2 if element]
df_grapes2 = pd.DataFrame(grapes2)
df_grapes2 = df_grapes2.drop_duplicates()
df_grapes2

,id,name,seo_name,has_detailed_info,wines_count,parent_grape_id
0,16,Sangiovese,sangiovese,True,125094,NaN
1,5,Chardonnay,chardonnay,True,604208,NaN
2,43,Corvina,corvina,False,22300,NaN
3,14,Pinot Noir,pinot-noir,True,572334,NaN
6,167,Tinta de toro,tinta-de-toro,False,3803,19.0
...,...,...,...,...,...,...
3790,40,Alicante Bouschet,alicante-bouschet,False,30014,NaN
3956,159,Pecorino,pecorino,False,5526,NaN
4061,111,Procanico,procanico,False,675,1556.0
4063,155,Muscat Ottonel,muscat-ottonel,False,4451,NaN


In [222]:
df_grapes2.to_csv(INTERIM_DATA / "vivino/grapes2_df.csv", index=False)

In [224]:
l = df_round2['grapes'].values
l = [[el[i]["id"] for i in range(len(el))] if el else np.nan for el in l]
df_round2['grapes'] = l

In [ ]:
# table of wine (parent)
wine2 = df_round2["wine"].values
df_wines2 = pd.json_normalize(wine2, sep='_')
df_wines2 = df_wines2.drop_duplicates(subset="id")
df_wines2

,id,name,seo_name,type_id,vintage_type,is_natural,vintage_mask,review_status,rank,hidden,...,specialists_notes,style_region,style,style_region_background_image,style_image_location,winery_dashboard_url,winery_background_image_location,winery_background_image_variations_large,winery_background_image_variations_medium,winery_background_image_variations_small
0,2774081,Júlio Bastos Grande Reserva,julio-bastos-grande-reserva,1,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100045,Morey-Saint-Denis Premier Cru Clos de La Bussière,morey-saint-denis-premier-cru-clos-de-la-bussiere,1,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2695604,Mocenni Numero 89 Gran Selezione Chianti Classico,mocenni-numero-89-gran-selezione-chianti-classico,1,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1981762,Terroir Series (Terroir Selection) Chardonnay,terroir-series-terroir-selection-chardonnay,2,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2716483,Pietro Dal Cero Amarone della Valpolicella,pietro-dal-cero-amarone-della-valpolicella,1,0,False,"[2024, 2025]",2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9755,2530402,La Casona de Arínzano,la-casona-de-arinzano,1,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,https://images.ctfassets.net/w1asd4g1aem0/4o5p...,NaN,NaN,NaN,NaN,NaN
9763,4136996,Vin de Days Rogue,vin-de-days-rogue,1,0,True,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9769,2581810,Zeltinger Sonnenuhr Riesling Auslese***,zeltinger-sonnenuhr-riesling-auslese,2,0,False,[2025],2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9774,1101056,Brunello di Montalcino,brunello-di-montalcino,1,0,False,"[2021, 2022, 2023, 2024, 2025]",2,None,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [230]:
df_wines2.to_csv(INTERIM_DATA / "vivino/wines2_df.csv", index=False)

In [233]:
# split col wine
l = df_round2['wine']
l = [el["id"] if el else np.nan for el in l]
df_round2['wine'] = l

In [235]:

# rating distributions
rat2 = df_round2['ratings_distribution']
rat2 = [[e[str(i)] for i in range(1, 6)] for e in rat2]
df_round2['ratings_distribution'] = rat2

In [237]:
l = df_round2["awards"].values
l = [subel for el in l for subel in el if subel]
df_awards2 = pd.json_normalize(l, sep='_')
df_awards2 = df_awards2.drop_duplicates(subset="ID")
df_awards2

,ID,CompetitionID,VintageID,Edition,Medal,Trophy,Score,Visible,URL,Origin,UserID,UpdatedAt,CreatedAt
0,15851,4,10746401,2016,Silver,None,None,True,https://www.iwsc.net/result/detail/69152,0,NaN,2022-10-19T12:39:30Z,2018-08-18T16:52:11Z
1,8132,3,109615568,2014,Silver,None,None,True,http://winejudging.com/medal_winners_2014/awar...,0,NaN,2022-10-19T12:37:26Z,2021-01-12T19:48:16Z
2,37602,3,109615568,2015,Gold,None,None,True,http://winejudging.com/medal_winners_2015/awar...,0,NaN,2022-10-19T12:45:02Z,2021-01-12T19:53:56Z
3,29797,2,11011645,2014,Gold,None,None,True,https://www.internationalwinechallenge.com/can...,0,NaN,2022-10-19T12:43:04Z,2018-03-15T12:25:33Z
4,150795,1,11047499,2017,Bronze,None,88,True,http://awards.decanter.com/DWWA/2017/Wine/3946...,0,NaN,2022-10-19T13:13:22Z,2018-03-23T13:11:50Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,40812,2,8090422,2015,Gold,None,None,True,https://www.internationalwinechallenge.com/can...,0,NaN,2022-10-19T12:45:50Z,2018-03-15T12:27:46Z
161,20408,3,8295607,2015,Double Gold,None,None,True,http://winejudging.com/medal_winners_2015/awar...,0,NaN,2022-10-19T12:40:39Z,2018-03-15T12:23:38Z
162,136286,2,86869236,2017,Silver,None,None,True,https://www.internationalwinechallenge.com/can...,0,NaN,2022-10-19T13:09:43Z,2018-03-23T13:04:21Z
163,32245,4,86869236,2017,Bronze,None,None,True,https://www.iwsc.net/result/detail/77429,0,NaN,2022-10-19T12:43:41Z,2018-08-18T16:53:25Z


In [250]:
df_awards2.to_csv(INTERIM_DATA / "vivino/df_awards2.csv", index=False)

In [ ]:
l = df_round2["awards"]
l = [[el[i]["ID"] for i in range(len(el))] if el else np.nan for el in l]
df_round2["awards"] = l

In [246]:
df_round2_wind = pd.json_normalize(df_round2['recommended_drinking_window'], sep='_').add_prefix("window_")
df_round2 = pd.concat([df_round2.drop(columns='recommended_drinking_window'), df_round2_wind], axis = 1)


In [ ]:
df_stats = pd.json_normalize(df_round2['statistics'].values, sep='_').add_prefix("stats_")
df_round2 = pd.concat([df_round2.drop(columns='statistics'), df_stats], axis = 1)


KeyError: 'statistics'

In [312]:
df_round2.to_csv(INTERIM_DATA / "vivino/df_round2.csv", index=False)

---

#### Fusion of the db

In [313]:
# test chekout prices
w1 = pd.read_csv(INTERIM_DATA / "vivino/df_round1.csv")
w2 = pd.read_csv(INTERIM_DATA / "vivino/df_round2.csv")

In [314]:
w = pd.concat([w1, w2])
w.to_csv(INTERIM_DATA / "vivino/df_all_wines.csv", index=False)

In [315]:
w1 = pd.read_csv(INTERIM_DATA / "vivino/awards1_df.csv")
w2 = pd.read_csv(INTERIM_DATA / "vivino/df_awards2.csv")
w = pd.concat([w1, w2])
w.to_csv(INTERIM_DATA / "vivino/df_all_awards.csv", index=False)

In [316]:
w1 = pd.read_csv(INTERIM_DATA / "vivino/grapes1_df.csv")
w2 = pd.read_csv(INTERIM_DATA / "vivino/grapes2_df.csv")
w = pd.concat([w1, w2])
w.to_csv(INTERIM_DATA / "vivino/df_all_grapes.csv", index=False)

In [317]:
w1 = pd.read_csv(INTERIM_DATA / "vivino/wines1_df.csv")
w2 = pd.read_csv(INTERIM_DATA / "vivino/wines2_df.csv")
w = pd.concat([w1, w2])
w.to_csv(INTERIM_DATA / "vivino/df_all_wines_parent.csv", index=False)

In [318]:
all_wines = pd.read_csv(INTERIM_DATA / "vivino/df_all_wines.csv")

In [323]:
all_wines['id'].values

array([100015094,  10013618,  10023048, ...,     99255,   9928081,
           99402])